In [2]:
import pandas as pd
import numpy as np
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Load datasets
diets_df = pd.read_csv('diets.csv')
medications_df = pd.read_csv('medications.csv')
precautions_df = pd.read_csv('precautions_df.csv')
symptom_severity_df = pd.read_csv('Symptom-severity.csv')
workout_df = pd.read_csv('workout_df.csv')

# Clean up the datasets as needed
diets_df['Diet'] = diets_df['Diet'].apply(lambda x: eval(x))  # Convert string representation of list to actual list
precautions_df = precautions_df.iloc[:, 1:].fillna('')  # Remove unnamed column and fill NaN values
medications_df['Medication'] = medications_df['Medication'].apply(lambda x: eval(x))  # Convert string representation of list to actual list

# Print dataset information
print("Data loaded successfully!")
print(f"Number of diseases with diet recommendations: {len(diets_df)}")
print(f"Number of diseases with medication recommendations: {len(medications_df)}")
print(f"Number of diseases with precaution recommendations: {len(precautions_df)}")
print(f"Number of symptoms with severity information: {len(symptom_severity_df)}")
print(f"Number of workout recommendations: {len(workout_df)}")

# Create a dictionary of symptoms and their severity
symptom_severity_dict = dict(zip(symptom_severity_df['Symptom'], symptom_severity_df['weight']))

# Group workout recommendations by disease
workout_recommendations = {}
for disease in workout_df['disease'].unique():
    workout_recommendations[disease] = workout_df[workout_df['disease'] == disease]['workout'].tolist()

# Function to get disease prediction based on symptoms
def predict_disease(symptoms_list):
    """
    Predicts disease based on user-entered symptoms

    Args:
        symptoms_list: List of symptoms entered by the user

    Returns:
        Most likely disease based on the symptoms
    """
    # For this beginner project, we'll use a simple approach based on matching symptoms
    # In a real project, you would use a trained machine learning model here

    # Calculate severity score based on symptoms
    severity_score = sum(symptom_severity_dict.get(symptom, 0) for symptom in symptoms_list)

    # For demonstration, we'll use a predefined mapping of symptoms to diseases
    # In a real project, this would be learned from training data
    # This is a simplified approach for the beginner project
    disease_matches = {
        "Fungal infection": ["itching", "skin_rash", "nodal_skin_eruptions"],
        "Allergy": ["continuous_sneezing", "shivering", "chills"],
        "GERD": ["stomach_pain", "acidity", "ulcers_on_tongue"],
        "Chronic cholestasis": ["itching", "vomiting", "yellowish_skin"],
        "Drug Reaction": ["itching", "skin_rash", "stomach_pain"],
        "Peptic ulcer disease": ["vomiting", "loss_of_appetite", "abdominal_pain"],
        "Diabetes": ["polyuria", "fatigue", "weight_loss", "excessive_hunger"],
        "Bronchial Asthma": ["breathlessness", "cough", "high_fever"],
        "Hypertension": ["headache", "chest_pain", "dizziness"],
        "Migraine": ["acidity", "indigestion", "headache"],
        "Jaundice": ["yellowish_skin", "dark_urine", "nausea"],
        "Malaria": ["chills", "vomiting", "high_fever"],
        "Chicken pox": ["itching", "skin_rash", "fatigue"],
        "Dengue": ["skin_rash", "chills", "joint_pain"],
        "Typhoid": ["chills", "vomiting", "high_fever", "headache", "constipation"],
        "Urinary tract infection": ["burning_micturition", "bladder_discomfort", "foul_smell_ofurine"],
        "Pneumonia": ["high_fever", "breathlessness", "sweating"],
    }

    # Count how many symptoms match each disease
    disease_scores = {}
    for disease, disease_symptoms in disease_matches.items():
        score = sum(1 for symptom in symptoms_list if symptom in disease_symptoms)
        if score > 0:  # Only include diseases with at least one matching symptom
            disease_scores[disease] = score

    # Return the disease with the highest score
    if disease_scores:
        return max(disease_scores, key=disease_scores.get)
    else:
        return "Unknown disease. Please consult a healthcare professional."

# Function to recommend medications based on predicted disease
def recommend_medications(disease):
    """
    Returns recommended medications for a disease

    Args:
        disease: Predicted disease

    Returns:
        List of recommended medications
    """
    if disease in medications_df['Disease'].values:
        return medications_df[medications_df['Disease'] == disease]['Medication'].values[0]
    else:
        return ["No specific medications found. Please consult a healthcare professional."]

# Function to recommend diet based on predicted disease
def recommend_diet(disease):
    """
    Returns recommended diet for a disease

    Args:
        disease: Predicted disease

    Returns:
        List of diet recommendations
    """
    if disease in diets_df['Disease'].values:
        return diets_df[diets_df['Disease'] == disease]['Diet'].values[0]
    else:
        return ["No specific diet recommendations found. Please consult a healthcare professional."]

# Function to recommend precautions based on predicted disease
def recommend_precautions(disease):
    """
    Returns recommended precautions for a disease

    Args:
        disease: Predicted disease

    Returns:
        List of precautions
    """
    if disease in precautions_df['Disease'].values:
        precautions = precautions_df[precautions_df['Disease'] == disease].iloc[0, 1:].tolist()
        return [p for p in precautions if p]  # Filter out empty strings
    else:
        return ["No specific precautions found. Please consult a healthcare professional."]

# Function to recommend workouts based on predicted disease
def recommend_workouts(disease):
    """
    Returns recommended workouts for a disease

    Args:
        disease: Predicted disease

    Returns:
        List of workout recommendations
    """
    if disease in workout_recommendations:
        return workout_recommendations[disease]
    else:
        return ["No specific workout recommendations found. Please consult a healthcare professional."]

# Main function to get recommendations based on symptoms
def get_recommendations(symptoms_list):
    """
    Main function to get recommendations based on symptoms

    Args:
        symptoms_list: List of symptoms entered by the user

    Returns:
        Dictionary containing disease prediction and recommendations
    """
    # Predict disease
    disease = predict_disease(symptoms_list)

    # Get recommendations
    recommendations = {
        "predicted_disease": disease,
        "medications": recommend_medications(disease),
        "diet": recommend_diet(disease),
        "precautions": recommend_precautions(disease),
        "workouts": recommend_workouts(disease)
    }

    return recommendations

# Simple UI for testing
def simple_console_ui():
    print("\n===== Medicine Recommendation System =====")
    print("Available symptoms:")

    # Display a subset of symptoms for user reference
    sample_symptoms = list(symptom_severity_dict.keys())[:20]
    for i, symptom in enumerate(sample_symptoms, 1):
        print(f"{i}. {symptom}")

    print("\nEnter symptoms (comma-separated) or type 'example' to use an example:")
    user_input = input()

    if user_input.lower() == 'example':
        symptoms = ["itching", "skin_rash", "nodal_skin_eruptions"]
        print(f"Using example symptoms: {symptoms}")
    else:
        symptoms = [s.strip() for s in user_input.split(',')]

    recommendations = get_recommendations(symptoms)

    print("\n===== Recommendations =====")
    print(f"Predicted Disease: {recommendations['predicted_disease']}")

    print("\nRecommended Medications:")
    for i, med in enumerate(recommendations['medications'], 1):
        print(f"{i}. {med}")

    print("\nRecommended Diet:")
    for i, diet in enumerate(recommendations['diet'], 1):
        print(f"{i}. {diet}")

    print("\nRecommended Precautions:")
    for i, precaution in enumerate(recommendations['precautions'], 1):
        print(f"{i}. {precaution}")

    print("\nRecommended Workouts/Lifestyle Changes:")
    for i, workout in enumerate(recommendations['workouts'][:5], 1):  # Limit to first 5 for readability
        print(f"{i}. {workout}")

    print("\nDISCLAIMER: This is a beginner project and not a substitute for professional medical advice.")

# Run the console UI
if __name__ == "__main__":
    simple_console_ui()

Data loaded successfully!
Number of diseases with diet recommendations: 41
Number of diseases with medication recommendations: 41
Number of diseases with precaution recommendations: 41
Number of symptoms with severity information: 133
Number of workout recommendations: 410

===== Medicine Recommendation System =====
Available symptoms:
1. itching
2. skin_rash
3. nodal_skin_eruptions
4. continuous_sneezing
5. shivering
6. chills
7. joint_pain
8. stomach_pain
9. acidity
10. ulcers_on_tongue
11. muscle_wasting
12. vomiting
13. burning_micturition
14. spotting_urination
15. fatigue
16. weight_gain
17. anxiety
18. cold_hands_and_feets
19. mood_swings
20. weight_loss

Enter symptoms (comma-separated) or type 'example' to use an example:

===== Recommendations =====
Predicted Disease: Unknown disease. Please consult a healthcare professional.

Recommended Medications:
1. No specific medications found. Please consult a healthcare professional.

Recommended Diet:
1. No specific diet recommendat